# Optima Life Product Portfolio Strategy

Evaluates a fictional five-product SaaS portfolio using ARR growth, renewal, engagement, and co-subscription behavior to recommend differentiated product strategies.

**Portfolio note:** This analysis was originally executed in a university-hosted Snowflake environment using a fictional SaaS dataset. The raw course dataset and hosted environment are not redistributed here. This notebook preserves the analytical workflow, SQL/Python code, methodology, and conclusions; the repository README summarizes the verified executed results.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown


pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


## 2. Build the Product Portfolio Dataset

The query below creates one product-level dataset containing:

- Customer count
- Renewal rate
- Average sessions from January 2023 through November 2024
- ARR for the two most recent annual snapshots in `ARR_ROW_YEAR`
- Absolute and percentage ARR growth

The latest ARR dates are selected directly from the existing Assignment 1 ARR table, so the growth period is determined by the available data rather than manually assigned.

In [ ]:
portfolio_query = r"""
WITH subscription_cohort AS (
    SELECT
        customer_id,
        product,
        renewed
    FROM ol_subscriptions
    WHERE subscription_end_date >= '2023-01-01'
      AND subscription_end_date < '2024-12-01'
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY customer_id, product
        ORDER BY subscription_end_date DESC,
                 subscription_start_date DESC
    ) = 1
),

daily_activity AS (
    SELECT
        customer_id,
        product,
        activity_date,
        SUM(num_sessions) AS daily_sessions
    FROM ol_customer_activity
    WHERE activity_date >= '2023-01-01'
      AND activity_date < '2024-12-01'
    GROUP BY
        customer_id,
        product,
        activity_date
),

customer_product_metrics AS (
    SELECT
        s.customer_id,
        s.product,
        s.renewed,
        COALESCE(SUM(a.daily_sessions), 0) AS total_sessions
    FROM subscription_cohort s
    LEFT JOIN daily_activity a
        ON s.customer_id = a.customer_id
       AND s.product = a.product
    GROUP BY
        s.customer_id,
        s.product,
        s.renewed
),

renewal_engagement AS (
    SELECT
        product,
        COUNT(*) AS customer_count,
        ROUND(
            100.0 * SUM(CASE WHEN renewed = TRUE THEN 1 ELSE 0 END)
            / NULLIF(COUNT(*), 0),
            2
        ) AS renewal_rate,
        ROUND(AVG(total_sessions), 1) AS avg_sessions
    FROM customer_product_metrics
    GROUP BY product
),

arr_by_product AS (
    SELECT
        product,
        year_dates,
        SUM(year_begin_arr) AS beginning_arr
    FROM arr_row_year
    GROUP BY product, year_dates
),

ranked_arr_dates AS (
    SELECT
        product,
        year_dates,
        beginning_arr,
        DENSE_RANK() OVER (
            ORDER BY year_dates DESC
        ) AS date_rank
    FROM arr_by_product
),

arr_growth AS (
    SELECT
        product,
        MAX(CASE WHEN date_rank = 2 THEN year_dates END) AS previous_arr_date,
        MAX(CASE WHEN date_rank = 1 THEN year_dates END) AS latest_arr_date,
        MAX(CASE WHEN date_rank = 2 THEN beginning_arr END) AS previous_arr,
        MAX(CASE WHEN date_rank = 1 THEN beginning_arr END) AS latest_arr
    FROM ranked_arr_dates
    WHERE date_rank <= 2
    GROUP BY product
)

SELECT
    r.product,
    r.customer_count,
    r.renewal_rate,
    r.avg_sessions,
    a.previous_arr_date,
    a.latest_arr_date,
    ROUND(a.previous_arr, 2) AS previous_arr,
    ROUND(a.latest_arr, 2) AS latest_arr,
    ROUND(a.latest_arr - a.previous_arr, 2) AS arr_growth,
    ROUND(
        100.0 * (a.latest_arr - a.previous_arr)
        / NULLIF(a.previous_arr, 0),
        2
    ) AS arr_growth_pct
FROM renewal_engagement r
LEFT JOIN arr_growth a
    ON r.product = a.product
ORDER BY arr_growth_pct DESC
"""

portfolio_df = session.sql(portfolio_query).to_pandas()

# Confirm the ARR period used by the query
previous_date = pd.to_datetime(portfolio_df["PREVIOUS_ARR_DATE"].dropna().iloc[0]).date()
latest_date = pd.to_datetime(portfolio_df["LATEST_ARR_DATE"].dropna().iloc[0]).date()

print(f"ARR growth period: {previous_date} to {latest_date}")
display(portfolio_df)

## 3. Compare Renewal Rates


In [ ]:
renewal_plot = portfolio_df.sort_values("RENEWAL_RATE")

plt.figure(figsize=(9, 5))
bars = plt.barh(
    renewal_plot["PRODUCT"],
    renewal_plot["RENEWAL_RATE"]
)

plt.title("Customer Renewal Rate by Product")
plt.xlabel("Renewal Rate (%)")
plt.ylabel("Product")
plt.xlim(0, 100)

for bar, value in zip(bars, renewal_plot["RENEWAL_RATE"]):
    plt.text(
        value + 0.8,
        bar.get_y() + bar.get_height() / 2,
        f"{value:.1f}%",
        va="center"
    )

plt.tight_layout()
plt.show()

## 4. Compare Customer Engagement



In [ ]:
engagement_plot = portfolio_df.sort_values("AVG_SESSIONS")

plt.figure(figsize=(9, 5))
bars = plt.barh(
    engagement_plot["PRODUCT"],
    engagement_plot["AVG_SESSIONS"]
)

plt.title("Average Customer Sessions by Product\nJan 2023 – Nov 2024")
plt.xlabel("Average Sessions")
plt.ylabel("Product")

for bar, value in zip(bars, engagement_plot["AVG_SESSIONS"]):
    plt.text(
        value + 0.5,
        bar.get_y() + bar.get_height() / 2,
        f"{value:.1f}",
        va="center"
    )

plt.tight_layout()
plt.show()

## 5. Compare Actual ARR Growth


In [ ]:
growth_plot = portfolio_df.sort_values("ARR_GROWTH_PCT")

plt.figure(figsize=(9, 5))
bars = plt.barh(
    growth_plot["PRODUCT"],
    growth_plot["ARR_GROWTH_PCT"]
)

plt.axvline(0, linewidth=1)
plt.title(
    f"ARR Growth by Product\n{previous_date} to {latest_date}"
)
plt.xlabel("ARR Growth (%)")
plt.ylabel("Product")

for bar, value in zip(bars, growth_plot["ARR_GROWTH_PCT"]):
    offset = 0.5 if value >= 0 else -0.5
    alignment = "left" if value >= 0 else "right"
    plt.text(
        value + offset,
        bar.get_y() + bar.get_height() / 2,
        f"{value:.1f}%",
        va="center",
        ha=alignment
    )

plt.tight_layout()
plt.show()

## 6. Position the Products Within the Portfolio

The portfolio matrix combines the three measures:

- The **horizontal position** shows actual ARR growth.
- The **vertical position** shows renewal rate.
- The **bubble size** shows average sessions.

The median ARR growth and renewal rate are used as relative portfolio benchmarks. These benchmarks do not label a product as objectively good or bad; they show how each product performs compared with the rest of the OptimaLife portfolio.

In [ ]:
chart_df = portfolio_df.dropna(
    subset=["ARR_GROWTH_PCT", "RENEWAL_RATE", "AVG_SESSIONS"]
).copy()

growth_benchmark = chart_df["ARR_GROWTH_PCT"].median()
renewal_benchmark = chart_df["RENEWAL_RATE"].median()
engagement_benchmark = chart_df["AVG_SESSIONS"].median()

# Bubble sizes reflect engagement, but engagement is treated as a supporting measure.
bubble_sizes = (
    chart_df["AVG_SESSIONS"]
    / chart_df["AVG_SESSIONS"].max()
) * 1300

plt.figure(figsize=(11, 7))

plt.scatter(
    chart_df["ARR_GROWTH_PCT"],
    chart_df["RENEWAL_RATE"],
    s=bubble_sizes,
    alpha=0.7,
    edgecolors="black",
    linewidth=1
)

for _, row in chart_df.iterrows():
    plt.annotate(
        row["PRODUCT"],
        (row["ARR_GROWTH_PCT"], row["RENEWAL_RATE"]),
        xytext=(7, 7),
        textcoords="offset points",
        fontsize=10
    )

plt.axvline(
    growth_benchmark,
    linestyle="--",
    linewidth=1,
    label=f"Median ARR growth: {growth_benchmark:.1f}%"
)

plt.axhline(
    renewal_benchmark,
    linestyle="--",
    linewidth=1,
    label=f"Median renewal: {renewal_benchmark:.1f}%"
)

plt.title(
    "OptimaLife Product Portfolio Positioning\n"
    "ARR Growth, Renewal and Engagement"
)
plt.xlabel("ARR Growth (%)")
plt.ylabel("Renewal Rate (%)")
plt.legend(loc="best")
plt.tight_layout()
plt.show()

print(f"Median ARR growth benchmark: {growth_benchmark:.2f}%")
print(f"Median renewal benchmark: {renewal_benchmark:.2f}%")
print(f"Median engagement benchmark: {engagement_benchmark:.1f} sessions")

## 7. Translate the Portfolio Matrix into Product Actions

The matrix shows that each product requires a different strategy:

- **Daily Fitness** is the strongest balanced performer. It has positive ARR growth, above-median renewal, and high engagement. It should be prioritized for continued investment and expansion.

- **Premium Health** has the strongest ARR growth but sits at the median renewal benchmark and has the lowest engagement. OptimaLife should continue pursuing growth while improving product adoption and retention.

- **Healthy Meals** has the highest renewal and engagement, but ARR declined slightly. It should be protected as a dependable core product and used for retention and cross-selling rather than treated as a major new-ARR growth product.

- **Wellness Tracker** falls below both the ARR-growth and renewal benchmarks. Its positioning and role within the portfolio should be reviewed before additional investment.

- **Mindful Living** has the weakest ARR growth and renewal performance. It is the highest-priority product for repositioning, bundling, or consolidation analysis.

The matrix therefore supports differentiated portfolio actions rather than applying one strategy across all five products. Engagement provides supporting context, while ARR direction and renewal performance remain the primary decision measures.

Note: The ARR growth percentage measures how much each product’s ARR changed compared with its own ARR one year earlier.



In [ ]:
def assign_portfolio_action(row):
    high_growth = row["ARR_GROWTH_PCT"] >= growth_benchmark
    high_renewal = row["RENEWAL_RATE"] >= renewal_benchmark
    high_engagement = row["AVG_SESSIONS"] >= engagement_benchmark

    if high_growth and high_renewal:
        if high_engagement:
            return "Prioritize investment and expansion"
        return "Invest selectively and strengthen engagement"

    if high_growth and not high_renewal:
        if high_engagement:
            return "Improve retention before further scaling"
        return "Strengthen engagement and retention before scaling"

    if not high_growth and high_renewal:
        return "Maintain as a core product and pursue cross-selling"

    if high_engagement:
        return "Review pricing or positioning despite active usage"

    return "Consider repositioning, bundling, or reduced scope"


portfolio_df["PORTFOLIO_ACTION"] = portfolio_df.apply(
    assign_portfolio_action,
    axis=1
)

portfolio_summary = portfolio_df[
    [
        "PRODUCT",
        "RENEWAL_RATE",
        "AVG_SESSIONS",
        "ARR_GROWTH_PCT",
        "PORTFOLIO_ACTION"
    ]
].sort_values("ARR_GROWTH_PCT", ascending=False)

display(portfolio_summary)

## 8. Product Portfolio Findings

The following cell converts the observed results into a concise business narrative. The statements are generated from the calculated values and do not depend on manually assigned product labels.

In [ ]:
strongest_growth = portfolio_df.loc[
    portfolio_df["ARR_GROWTH_PCT"].idxmax()
]
strongest_renewal = portfolio_df.loc[
    portfolio_df["RENEWAL_RATE"].idxmax()
]
strongest_engagement = portfolio_df.loc[
    portfolio_df["AVG_SESSIONS"].idxmax()
]

weakest_growth = portfolio_df.loc[
    portfolio_df["ARR_GROWTH_PCT"].idxmin()
]
weakest_renewal = portfolio_df.loc[
    portfolio_df["RENEWAL_RATE"].idxmin()
]
weakest_engagement = portfolio_df.loc[
    portfolio_df["AVG_SESSIONS"].idxmin()
]

engagement_range = (
    portfolio_df["AVG_SESSIONS"].max()
    - portfolio_df["AVG_SESSIONS"].min()
)

findings = f"""
### Main Findings

**{strongest_growth['PRODUCT']}** has the strongest recent ARR growth at
**{strongest_growth['ARR_GROWTH_PCT']:.1f}%**, making it the portfolio's
strongest financial growth candidate.

**{strongest_renewal['PRODUCT']}** has the highest renewal rate at
**{strongest_renewal['RENEWAL_RATE']:.1f}%**, while
**{strongest_engagement['PRODUCT']}** has the highest average engagement at
**{strongest_engagement['AVG_SESSIONS']:.1f} sessions**.

Engagement varies by only **{engagement_range:.1f} sessions** from the highest
to the lowest product. Therefore, engagement provides supporting context but
does not separate the portfolio as clearly as renewal and ARR growth.

At the weaker end of the portfolio, **{weakest_growth['PRODUCT']}** has the
lowest recent ARR growth at **{weakest_growth['ARR_GROWTH_PCT']:.1f}%**,
**{weakest_renewal['PRODUCT']}** has the lowest renewal rate at
**{weakest_renewal['RENEWAL_RATE']:.1f}%**, and
**{weakest_engagement['PRODUCT']}** has the lowest average engagement at
**{weakest_engagement['AVG_SESSIONS']:.1f} sessions**.

The portfolio should therefore be managed through differentiated actions rather
than one common strategy. Products with strong ARR growth and renewal justify
continued investment. Products that are growing but retain customers poorly
need retention improvements before aggressive scaling. Products with strong
renewal but slower growth should be maintained as dependable core offerings and
used for cross-selling. Products that trail the portfolio in both growth and
renewal warrant a review of positioning, bundling, investment, or scope.
"""

display(Markdown(findings))

## 9. Test Dropping Versus Bundling Underperforming Products

Co-subscription means that the customer held both products at the same ARR snapshot, rather than purchasing them at different times.

In [ ]:
# Identify underperformers from the portfolio results
underperformers = portfolio_df.loc[
    (portfolio_df["ARR_GROWTH_PCT"] < 0)
    & (portfolio_df["RENEWAL_RATE"] < renewal_benchmark),
    "PRODUCT"
].tolist()

if not underperformers:
    raise ValueError("No products meet the underperformer rule.")

candidate_values = ", ".join(
    "'" + product.replace("'", "''") + "'"
    for product in underperformers
)


co_subscription_query = f"""
WITH active_customer_product AS (
    SELECT
        customer_id,
        product,
        SUM(row_arr) AS current_arr
    FROM arr_row_level
    WHERE subscription_start_date <= '{latest_date}'
      AND subscription_end_date > '{latest_date}'
    GROUP BY
        customer_id,
        product
),

candidate_base AS (
    SELECT
        product AS candidate_product,
        customer_id,
        current_arr AS candidate_arr
    FROM active_customer_product
    WHERE product IN ({candidate_values})
),

customer_total_arr AS (
    SELECT
        customer_id,
        SUM(current_arr) AS customer_portfolio_arr
    FROM active_customer_product
    GROUP BY customer_id
),

candidate_customer_flags AS (
    SELECT
        c.candidate_product,
        c.customer_id,
        c.candidate_arr,
        COUNT(p.product) AS other_product_count
    FROM candidate_base c

    LEFT JOIN active_customer_product p
        ON c.customer_id = p.customer_id
       AND c.candidate_product <> p.product

    GROUP BY
        c.candidate_product,
        c.customer_id,
        c.candidate_arr
),

candidate_summary AS (
    SELECT
        candidate_product,

        COUNT(*) AS candidate_total_customers,

        SUM(candidate_arr) AS candidate_total_arr,

        COUNT_IF(
            other_product_count > 0
        ) AS co_sub_customers,

        ROUND(
            100.0
            * COUNT_IF(other_product_count > 0)
            / NULLIF(COUNT(*), 0),
            2
        ) AS co_sub_rate_pct,

        SUM(
            IFF(
                other_product_count > 0,
                candidate_arr,
                0
            )
        ) AS co_sub_candidate_arr,

        COUNT_IF(
            other_product_count = 0
        ) AS solo_customers,

        SUM(
            IFF(
                other_product_count = 0,
                candidate_arr,
                0
            )
        ) AS solo_candidate_arr

    FROM candidate_customer_flags

    GROUP BY candidate_product
),

pair_summary AS (
    SELECT
        c.candidate_product,
        p.product AS partner_product,

        COUNT(*) AS shared_customers,

        SUM(
            c.candidate_arr
        ) AS shared_candidate_arr,

        SUM(
            t.customer_portfolio_arr
        ) AS shared_customer_portfolio_arr

    FROM candidate_base c

    INNER JOIN active_customer_product p
        ON c.customer_id = p.customer_id
       AND c.candidate_product <> p.product

    INNER JOIN customer_total_arr t
        ON c.customer_id = t.customer_id

    GROUP BY
        c.candidate_product,
        p.product
)

SELECT
    p.candidate_product,
    p.partner_product,

    s.candidate_total_customers,

    ROUND(
        s.candidate_total_arr,
        2
    ) AS candidate_total_arr,

    s.co_sub_customers,
    s.co_sub_rate_pct,

    ROUND(
        s.co_sub_candidate_arr,
        2
    ) AS co_sub_candidate_arr,

    s.solo_customers,

    ROUND(
        s.solo_candidate_arr,
        2
    ) AS solo_candidate_arr,

    p.shared_customers,

    ROUND(
        100.0
        * p.shared_customers
        / NULLIF(s.candidate_total_customers, 0),
        2
    ) AS partner_overlap_pct,

    ROUND(
        p.shared_candidate_arr,
        2
    ) AS shared_candidate_arr,

    ROUND(
        p.shared_customer_portfolio_arr,
        2
    ) AS shared_customer_portfolio_arr

FROM pair_summary p

INNER JOIN candidate_summary s
    ON p.candidate_product = s.candidate_product

ORDER BY
    p.candidate_product,
    p.shared_candidate_arr DESC
"""


co_subscription_df = session.sql(
    co_subscription_query
).to_pandas()


# Candidate-level drop exposure
candidate_summary_df = (
    co_subscription_df[
        [
            "CANDIDATE_PRODUCT",
            "CANDIDATE_TOTAL_CUSTOMERS",
            "CANDIDATE_TOTAL_ARR",
            "CO_SUB_CUSTOMERS",
            "CO_SUB_RATE_PCT",
            "CO_SUB_CANDIDATE_ARR",
            "SOLO_CUSTOMERS",
            "SOLO_CANDIDATE_ARR"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        "CANDIDATE_TOTAL_ARR",
        ascending=False
    )
)


# Partner-level bundling opportunities
pair_summary_df = (
    co_subscription_df[
        [
            "CANDIDATE_PRODUCT",
            "PARTNER_PRODUCT",
            "SHARED_CUSTOMERS",
            "PARTNER_OVERLAP_PCT",
            "SHARED_CANDIDATE_ARR",
            "SHARED_CUSTOMER_PORTFOLIO_ARR"
        ]
    ]
    .sort_values(
        [
            "CANDIDATE_PRODUCT",
            "SHARED_CANDIDATE_ARR"
        ],
        ascending=[True, False]
    )
)


display(
    Markdown(
        "### Underperformer Exposure and Co-Subscription Summary"
    )
)

display(candidate_summary_df)


display(
    Markdown(
        "### Partner-Level Bundling Opportunities"
    )
)

display(pair_summary_df)

## 10. Visualize the Strongest Co-Subscription Relationships


In [ ]:
plot_df = pair_summary_df.sort_values(
    "PARTNER_OVERLAP_PCT"
).copy()

plot_df["PAIR"] = (
    plot_df["CANDIDATE_PRODUCT"]
    + " + "
    + plot_df["PARTNER_PRODUCT"]
)

plt.figure(figsize=(10, 6))

bars = plt.barh(
    plot_df["PAIR"],
    plot_df["PARTNER_OVERLAP_PCT"]
)

plt.title(
    "Active Co-Subscription Rate by Candidate and Partner\n"
    "December 2024 Snapshot"
)
plt.xlabel(
    "Candidate Customers Also Holding Partner Product (%)"
)
plt.ylabel("Candidate + Partner")

for bar, value in zip(
    bars,
    plot_df["PARTNER_OVERLAP_PCT"]
):
    plt.text(
        value + 0.4,
        bar.get_y() + bar.get_height() / 2,
        f"{value:.1f}%",
        va="center"
    )

plt.tight_layout()
plt.show()

Mindful Living and Wellness Tracker are the clearest underperforming products because both have negative ARR growth and below-median renewal. However, the co-subscription results do not support dropping either product immediately.

63% of Mindful Living and Wellness Tracker customers also subscribe to at least one other OptimaLife product. Dropping either product would therefore put a large amount of existing customer value at risk and could affect relationships that extend across the broader portfolio.

Because Premium Health has the strongest ARR growth and the largest existing customer overlap with both underperforming products, it is the most logical option for an initial bundling. 

## Final Recommendation

OptimaLife should manage the portfolio through differentiated product strategies:

1. **Daily Fitness — prioritize investment and expansion.** It is the strongest balanced product, combining 3.2% ARR growth with 77.8% renewal and 63.2 average sessions.

2. **Premium Health — continue pursuing new ARR while strengthening adoption and retention.** Its 28.0% ARR growth is the strongest in the portfolio, but its 66.0% renewal and 47.0 average sessions show that growth is not yet supported by equally strong retention and engagement.

3. **Healthy Meals — protect as the core retention and cross-sell product.** It leads renewal at 78.1% and engagement at 63.9 sessions. Its 1.9% ARR decline should be investigated, but the product remains a dependable foundation for the portfolio.

4. **Wellness Tracker — reposition and test bundling before reducing its scope.** Its declining ARR and below-median renewal make it an underperformer, but approximately 63% of its customers also hold another product. Dropping it immediately could put substantial existing customer value at risk.

5. **Mindful Living — make it the highest-priority turnaround and consolidation candidate.** It has the weakest ARR growth and renewal, but its substantial co-subscription rate means that immediate removal could affect valuable multi-product customer relationships.

The recommended first step is to test bundling or feature consolidation with Premium Health, which has the strongest customer overlap with both underperforming products. OptimaLife should follow a **bundle first, evaluate the results, and drop only if the pilot fails** strategy.
